# MedPilot - vLLM + Qwen2.5-1.5B on Colab

**Runtime -> GPU (T4 or better)**

vLLM cho tốc độ phản hồi nhanh hơn transformers thuần.

In [ ]:
# Install vLLM
!pip install vllm pyngrok fastapi uvicorn -q

In [ ]:
import subprocess
import sys

# Kill any existing processes on port 8000
subprocess.run("pkill -f 'uvicorn' || true", shell=True)
subprocess.run("fuser -k 8000/tcp || true", shell=True)

print("Ports cleared!")

In [ ]:
import os
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

from vllm import LLM, SamplingParams
from pyngrok import ngrok
import threading
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional

# Ngrok token - đăng ký free tại ngrok.com
NGROK_TOKEN = "3BJuRXeiwgSNG5zWfm89ESHTDtg_3QBZ3CU3f7doQ3a6stpQS"
ngrok.set_auth_token(NGROK_TOKEN)

In [ ]:
# Load model với vLLM
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print("🚀 Loading Qwen2.5-1.5B with vLLM...")
print("   This may take 3-5 minutes on first run")

llm = LLM(
    model=MODEL_NAME,
    tensor_parallel_size=1,
    max_model_len=4096,
    gpu_memory_utilization=0.85,
    dtype="float16"
)

print("✅ Model loaded!")

In [ ]:
# Test generation
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Test prompt
test_prompt = "Xin chào, bạn là ai?"
messages = [{"role": "user", "content": test_prompt}]
prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_dict=True, return_tensors="pt")["input_ids"]

sampling_params = SamplingParams(temperature=0.7, max_tokens=100)
output = llm.generate([prompt], sampling_params)[0].outputs[0].text

print(f"Test output: {output}")

In [ ]:
# FastAPI app - OpenAI compatible
app = FastAPI()

class Message(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    messages: List[Message]
    max_tokens: Optional[int] = 1000
    temperature: Optional[float] = 0.7

@app.post("/v1/chat/completions")
async def chat_completions(data: ChatRequest):
    try:
        # Extract messages
        messages = data.messages
        
        # Build prompt with chat template
        prompt = tokenizer.apply_chat_template(
            messages, 
            add_generation_prompt=True, 
            return_dict=True, 
            return_tensors="pt"
        )["input_ids"]
        
        # Sampling params
        sampling_params = SamplingParams(
            temperature=data.temperature or 0.7,
            max_tokens=data.max_tokens or 1000,
            stop=["<|im_end|>"]
        )
        
        # Generate
        outputs = llm.generate([prompt], sampling_params)
        response_text = outputs[0].outputs[0].text
        
        return {
            "id": "chatcmpl-" + str(hash(str(messages)))[:8],
            "object": "chat.completion",
            "created": 1234567890,
            "model": MODEL_NAME,
            "choices": [{
                "index": 0,
                "message": {
                    "role": "assistant",
                    "content": response_text
                },
                "finish_reason": "stop"
            }],
            "usage": {
                "prompt_tokens": prompt.shape[1],
                "completion_tokens": len(tokenizer.encode(response_text)),
                "total_tokens": prompt.shape[1] + len(tokenizer.encode(response_text))
            }
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/")
def root():
    return {"message": "MedPilot vLLM Server - Qwen2.5-1.5B", "status": "running"}

@app.get("/health")
def health():
    return {"status": "healthy"}

In [ ]:
# Start ngrok + server
public_url = ngrok.connect(8000, bind_tls=True)
print(f"🌐 Public URL: {public_url}")
print(f"📍 Endpoint: {public_url}/v1/chat/completions")

# Update .env with this URL
print("\n" + "="*60)
print("🔧 Update your .env file:")
print(f"VLLM_API_URL={public_url}/v1/chat/completions")
print("="*60)

In [ ]:
# Run server
def run_server():
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    server.run()

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("✅ Server is running!")
print("⏰ Keep this tab open!")